# Progetto B1: Explainable AI & Adversarial Attacks

- Obiettivo: Addestrare una CNN custom e dimostrare di saper manipolare i gradienti sia per spiegare cosa la rete sta imparando (Grad-CAM), sia per tentare di ingannarla (Adversarial Attack).

### Specifiche:
1. Training: Addestrare una CNN (definita da voi, o pre-trainata) su un dataset come TinyImageNet, CIFAR-10 o subset di animali.
- Addestrate il sistema fino a convergenza.
- Mostrate le curve di apprendimento e risolvete ogni problema di underfitting, o overfitting. Presentare un sistema in cui gli addestramenti non sono completi è considerato come non averlo addestrato per nulla, in quanto non si possono fare considerazioni conclusive.
- Sicuramente avrete degli iper-parametri: progettate accuratamente gli esperimenti di validazione tendo conto della dimensione della rete, del numero di sample del dataset e altre condizioni al contorno. Le scelte progettuali non possono essere arbitrarie.
2. Grad-CAM: Implementare manualmente l'algoritmo Grad-CAM per visualizzare le heatmap di attivazione sulla classe predetta.
3. Attacco FGSM: Implementare il Fast Gradient Sign Method. Generare un adversarial example aggiungendo rumore impercettibile calcolato tramite il gradiente della loss rispetto all'input.
4. Analisi:
- Scegliere le metriche più opportune per testare il sistema
- Considerate gli attacchi alla vostra rete come attacchi “white-box”. Questa è la vostra baseline iniziale. Quindi esplorate l’efficacia di attacchi black-box su reti diverse addestrate per lo stesso task, o per task simili.
- Proporre, sulla base dell’analisi iniziale, strategie di miglioramento della strategia di attacco. Ipotizzare possibili contromisure e testarne l’efficacia.
- Visualizzare fianco a fianco: Immagine Originale + CAM, Immagine Avversaria + CAM. Commentare come cambia l'attenzione della rete.

### Hints Step-by-Step:
- Architettura suggerita: 4-5 blocchi Conv-ReLU-MaxPool seguiti da Global Average Pooling (importante per Grad-CAM) e un Linear layer.
- Grad-CAM: Usate register_forward_hook e register_backward_hook (o accedete a .grad) sull'ultimo layer convoluzionale per catturare feature maps () e gradienti ().
- FGSM: La formula è . Ricordate di fare input.requires_grad = True prima del forward pass per calcolare il gradiente rispetto all'immagine.

# Implementazione

## 1. Definisco una rete CNN e la addestro sul dataset CIFAR-10

## 1. Importare librerie
Importo le librerie Pythorch e Torchvision


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

## 2. Configurare il Device
Se disponibile uso la GPU per rendere più veloce l'addestramento


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


## 3. Caricamento e Preprocess Dataset
Creazione pipeline di Data augmentation e normalizzazione dei dati


In [3]:
transform_train_val = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])


Importazione trasformazione e suddivisione del dataset in train, validation e test

In [4]:
trainset_full = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transform_train_val
)

testset = torchvision.datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=transform_test
)

# utils per lo suddivisione random
from torch.utils.data import random_split

# Suddivisione in train e validation
train_size = int(0.8 * len(trainset_full))
val_size = len(trainset_full) - train_size

train_dataset, val_dataset = random_split(
    trainset_full,
    [train_size, val_size]
)

trainloader = DataLoader(train_dataset, batch_size=128, shuffle=True)
valloader = DataLoader(val_dataset, batch_size=128, shuffle=False)

testloader = DataLoader(testset, batch_size=128, shuffle=False)

print("Train size:", len(train_dataset))
print("Validation size:", len(val_dataset))
print("Test size:", len(testset))

100%|██████████| 170M/170M [01:11<00:00, 2.37MB/s] 


Train size: 40000
Validation size: 10000
Test size: 10000
